In [1]:
from datetime import datetime, timedelta
from email.utils import parsedate_to_datetime
from urllib.parse import quote_plus
from zoneinfo import ZoneInfo

import feedparser
import requests

In [ ]:
SOURCE_CANDIDATES = {
    # 1순위: 미국 증시/금융 전문
    "finance.yahoo.com": "Yahoo Finance",
    "cnbc.com": "CNBC",
    "marketwatch.com": "MarketWatch",
    "barrons.com": "Barron's",

    # 2순위: 경제/비즈니스 종합
    "fortune.com": "Fortune",
    "forbes.com": "Forbes",
    "businessinsider.com": "Business Insider",
    "cnn.com": "CNN Business",
    "foxbusiness.com": "Fox Business",
    "apnews.com": "Associated Press",

    # 3순위: 투자 전문
    "benzinga.com": "Benzinga",
    "thestreet.com": "TheStreet",
    "fool.com": "The Motley Fool",
    "seekingalpha.com": "Seeking Alpha",
    "investing.com": "Investing.com",
    "zacks.com": "Zacks",

    # 접근 제한 가능성이 높은 곳도 비교용으로 테스트
    "bloomberg.com": "Bloomberg",
    "wsj.com": "Wall Street Journal",
    "ft.com": "Financial Times",
    "reuters.com": "Reuters",
}


TEST_QUERIES = [
    "stock market",
    "Wall Street",
    "Federal Reserve",
    "earnings",
    "NVIDIA",
]

START_DATE = "2026-10-20"
END_DATE = "2026-10-21"

NEWS_TIMEZONE = ZoneInfo("America/New_York")

In [3]:
def date_range(start_date, end_date):
    start = datetime.strptime(start_date, "%Y-%m-%d").date()
    end = datetime.strptime(end_date, "%Y-%m-%d").date()

    current = start

    while current <= end:
        yield current
        current += timedelta(days=1)

In [4]:
def build_google_news_rss_url(query, source, target_date):
    """
    target_date 주변을 조금 넓게 검색한 뒤
    실제 날짜 필터링은 Python에서 다시 수행합니다.
    """

    search_after = target_date - timedelta(days=1)
    search_before = target_date + timedelta(days=1)

    search_query = (
        f'"{query}" '
        f'site:{source} '
        f'after:{search_after.isoformat()} '
        f'before:{search_before.isoformat()}'
    )

    encoded_query = quote_plus(search_query)

    url = (
        "https://news.google.com/rss/search"
        f"?q={encoded_query}"
        "&hl=en-US"
        "&gl=US"
        "&ceid=US:en"
    )

    return url, search_query

In [5]:
def search_google_news(query, source, target_date):
    url, search_query = build_google_news_rss_url(
        query=query,
        source=source,
        target_date=target_date,
    )

    response = requests.get(
        url,
        headers={
            "User-Agent": "Mozilla/5.0"
        },
        timeout=15,
    )

    response.raise_for_status()

    feed = feedparser.parse(response.content)

    articles = []

    for entry in feed.entries:

        published_raw = entry.get("published")

        if not published_raw:
            continue

        try:
            published_dt = parsedate_to_datetime(published_raw)

            # RSS 시간은 보통 GMT이므로
            # 미국 동부시간으로 변환
            published_et = published_dt.astimezone(NEWS_TIMEZONE)

        except Exception:
            continue

        # 우리가 원하는 미국 동부시간 날짜와 정확히 일치하는지 검사
        if published_et.date() != target_date:
            continue

        rss_source = None

        if hasattr(entry, "source"):
            rss_source = entry.source.get("title")

        articles.append({
            "query": query,
            "requested_source": source,

            "title": entry.get("title"),
            "google_news_url": entry.get("link"),

            "source": rss_source,

            "published_raw": published_raw,
            "published_at": published_dt.isoformat(),
            "published_at_et": published_et.isoformat(),

            "target_date": target_date.isoformat(),

            "search_query": search_query,
        })

    return articles

In [6]:
def collect_google_news(
    search_terms,
    sources,
    start_date,
    end_date,
):
    all_articles = []

    dates = list(date_range(start_date, end_date))

    total = (
        len(search_terms)
        * len(sources)
        * len(dates)
    )

    count = 0

    for target_date in dates:

        for query in search_terms:

            for source in sources:
                count += 1

                print(
                    f"[{count}/{total}] "
                    f"{target_date} | "
                    f"{source} | "
                    f"{query}"
                )

                try:
                    articles = search_google_news(
                        query=query,
                        source=source,
                        target_date=target_date,
                    )

                    print(
                        f"    → {len(articles)} articles"
                    )

                    all_articles.extend(articles)

                except Exception as e:
                    print(
                        f"    → ERROR: "
                        f"{type(e).__name__}: {e}"
                    )

    return all_articles

In [7]:
articles = collect_google_news(
    search_terms=TEST_QUERIES,
    sources=SOURCE_CANDIDATES,
    start_date=START_DATE,
    end_date=END_DATE,
)

len(articles)

[1/200] 2026-10-20 | finance.yahoo.com | stock market
    → 66 articles
[2/200] 2026-10-20 | cnbc.com | stock market
    → 36 articles
[3/200] 2026-10-20 | marketwatch.com | stock market
    → 28 articles
[4/200] 2026-10-20 | barrons.com | stock market
    → 24 articles
[5/200] 2026-10-20 | fortune.com | stock market
    → 4 articles
[6/200] 2026-10-20 | forbes.com | stock market
    → 2 articles
[7/200] 2026-10-20 | businessinsider.com | stock market
    → 42 articles
[8/200] 2026-10-20 | cnn.com | stock market
    → 3 articles
[9/200] 2026-10-20 | foxbusiness.com | stock market
    → 7 articles
[10/200] 2026-10-20 | apnews.com | stock market
    → 2 articles
[11/200] 2026-10-20 | benzinga.com | stock market
    → 62 articles
[12/200] 2026-10-20 | thestreet.com | stock market
    → 41 articles
[13/200] 2026-10-20 | fool.com | stock market
    → 72 articles
[14/200] 2026-10-20 | seekingalpha.com | stock market
    → 39 articles
[15/200] 2026-10-20 | investing.com | stock market
    → 5

6748

In [8]:
def remove_duplicates(articles):
    unique_articles = {}

    for article in articles:

        key = (
            article["title"],
            article["source"],
            article["published_at"],
        )

        if key not in unique_articles:
            unique_articles[key] = article

        else:
            # 같은 기사가 여러 검색어에 걸린 경우
            existing = unique_articles[key]

            if "matched_queries" not in existing:
                existing["matched_queries"] = [
                    existing["query"]
                ]

            if article["query"] not in existing["matched_queries"]:
                existing["matched_queries"].append(
                    article["query"]
                )

    return list(unique_articles.values())

In [9]:
articles = remove_duplicates(articles)

print(f"중복 제거 후: {len(articles)}")

중복 제거 후: 4942


In [18]:
#pip install -U googlenewsdecoder

In [10]:
from googlenewsdecoder import gnews_decoder_async


async def resolve_original_urls(
    articles,
    batch_size=50,
    concurrency=6,
):
    resolved_articles = []

    for start in range(0, len(articles), batch_size):
        batch = articles[start:start + batch_size]

        google_urls = [
            article["google_news_url"]
            for article in batch
        ]

        try:
            results = await gnews_decoder_async(
                google_urls,
                interval=0.2,
                timeout=15.0,
                concurrency=concurrency,
            )

        except Exception as e:
            # 배치 전체 실패
            for article in batch:
                resolved_articles.append({
                    **article,
                    "original_url": None,
                    "resolve_status": "ERROR",
                    "resolve_error": f"{type(e).__name__}: {e}",
                })

            continue

        for article, result in zip(batch, results):

            if result.get("success"):
                resolved_articles.append({
                    **article,
                    "original_url": result["decoded_url"],
                    "resolve_status": "SUCCESS",
                    "resolve_error": None,
                })

            else:
                resolved_articles.append({
                    **article,
                    "original_url": None,
                    "resolve_status": "FAILED",
                    "resolve_error": result.get("message"),
                })

    return resolved_articles

In [11]:
resolved_articles = await resolve_original_urls(
    articles,
    batch_size=50,
    concurrency=6,
)

In [12]:
for article in resolved_articles[:10]:
    print(
        article["resolve_status"],
        "|",
        article["original_url"]
    )

SUCCESS | https://finance.yahoo.com/healthcare/articles/moderna-jumps-7-while-melanoma-175326781.html
SUCCESS | https://finance.yahoo.com/news/paul-tudor-jones-makes-chilling-154500005.html
SUCCESS | https://finance.yahoo.com/economy/article/advice-for-home-sellers-if-youre-trying-to-time-the-market-you-cant-123147322.html
SUCCESS | https://finance.yahoo.com/markets/stocks/articles/stock-market-today-oct-20-220001509.html
SUCCESS | https://finance.yahoo.com/markets/stocks/articles/stock-market-today-oct-20-213835202.html
SUCCESS | https://finance.yahoo.com/markets/stocks/articles/united-parcel-ups-beats-stock-210006260.html
SUCCESS | https://finance.yahoo.com/markets/stocks/articles/qualcomm-qcom-beats-stock-market-204505184.html
SUCCESS | https://finance.yahoo.com/markets/stocks/articles/warren-buffett-issuing-stark-warning-082000676.html
SUCCESS | https://finance.yahoo.com/markets/stocks/articles/stock-market-today-oct-20-210942173.html
SUCCESS | https://finance.yahoo.com/markets/sto

In [13]:
for i in resolved_articles[:3]:
    print(i)

{'query': 'stock market', 'requested_source': 'finance.yahoo.com', 'title': 'Moderna Jumps 7% While Melanoma Vaccine Data Looms; Bristol Myers Squibb Rises 5%, Pfizer Advances 2% - Yahoo Finance', 'google_news_url': 'https://news.google.com/rss/articles/CBMilgFBVV95cUxOMlg3YXlXcTUzYUxzbXRkSlRJUUVTRXJ5LW1KMS1GZ1J1SDVXR1BIdF9TM2pjaFljUXZUdHZORm50YjhaVnRNSXNvQ1JYLXhKRGExbk1YeE9GZWxoeXg0aktmZW1vbE5mTXA3a1RhRXZWYlVUbFJZbnE1VVFsVTVyRFZwazUxYS05MVdhU082NE9pejdYNXc?oc=5', 'source': 'Yahoo Finance', 'published_raw': 'Tue, 20 Oct 2026 17:53:26 GMT', 'published_at': '2026-10-20T17:53:26+00:00', 'published_at_et': '2026-10-20T13:53:26-04:00', 'target_date': '2026-10-20', 'search_query': '"stock market" site:finance.yahoo.com after:2026-10-19 before:2026-10-21', 'original_url': 'https://finance.yahoo.com/healthcare/articles/moderna-jumps-7-while-melanoma-175326781.html', 'resolve_status': 'SUCCESS', 'resolve_error': None}
{'query': 'stock market', 'requested_source': 'finance.yahoo.com', 'title': '

In [14]:
import random
import requests
import pandas as pd

from collections import defaultdict
from trafilatura import extract

In [15]:
MIN_TEST_ARTICLES = 5
PASS_RATE = 0.8

In [16]:
def classify_source(total, success):
    if total < MIN_TEST_ARTICLES:
        return "INSUFFICIENT_SAMPLE"

    success_rate = success / total

    if success_rate >= 0.8:
        return "USABLE"

    if success_rate >= 0.5:
        return "UNSTABLE"

    return "NOT_USABLE"

In [17]:
def sample_articles_for_test(
    resolved_articles,
    per_query=10,
    seed=42,
):
    groups = defaultdict(list)

    # source × query별 그룹 생성
    for article in resolved_articles:
        source = article.get("requested_source")
        query = article.get("query")
        url = article.get("original_url")

        if not source or not query or not url:
            continue

        groups[(source, query)].append(article)

    rng = random.Random(seed)

    sampled_articles = []

    for (source, query), articles in groups.items():

        # original_url 기준 중복 제거
        unique = {}

        for article in articles:
            url = article["original_url"]

            if url not in unique:
                unique[url] = article

        candidates = list(unique.values())

        # 최대 10개 랜덤 추출
        if len(candidates) > per_query:
            candidates = rng.sample(
                candidates,
                per_query,
            )

        sampled_articles.extend(candidates)

        print(
            f"{source:25s} | "
            f"{query:20s} | "
            f"{len(candidates)}개"
        )

    return sampled_articles

In [18]:
test_articles = sample_articles_for_test(
    resolved_articles,
    per_query=10,
    seed=42,
)

print()
print(f"총 테스트 기사 수: {len(test_articles)}")

finance.yahoo.com         | stock market         | 10개
cnbc.com                  | stock market         | 10개
marketwatch.com           | stock market         | 10개
barrons.com               | stock market         | 10개
fortune.com               | stock market         | 8개
forbes.com                | stock market         | 8개
businessinsider.com       | stock market         | 10개
cnn.com                   | stock market         | 6개
foxbusiness.com           | stock market         | 10개
apnews.com                | stock market         | 7개
benzinga.com              | stock market         | 10개
thestreet.com             | stock market         | 10개
fool.com                  | stock market         | 10개
seekingalpha.com          | stock market         | 10개
investing.com             | stock market         | 10개
zacks.com                 | stock market         | 10개
bloomberg.com             | stock market         | 10개
wsj.com                   | stock market         | 10개
ft.com        

In [19]:
def crawl_article_for_test(
    article,
    session,
    timeout=15,
    min_body_length=300,
):
    url = article["original_url"]

    try:
        response = session.get(
            url,
            timeout=timeout,
            allow_redirects=True,
        )

    except requests.Timeout:
        return {
            "crawl_status": "TIMEOUT",
            "http_status": None,
            "body_length": 0,
            "body": None,
            "crawl_error": "Request timeout",
        }

    except requests.RequestException as e:
        return {
            "crawl_status": "REQUEST_ERROR",
            "http_status": None,
            "body_length": 0,
            "body": None,
            "crawl_error": f"{type(e).__name__}: {e}",
        }

    status = response.status_code

    # HTTP 오류
    if status != 200:
        return {
            "crawl_status": f"HTTP_{status}",
            "http_status": status,
            "body_length": 0,
            "body": None,
            "crawl_error": f"HTTP {status}",
        }

    # HTML이 아닌 경우
    content_type = response.headers.get(
        "Content-Type",
        "",
    ).lower()

    if "html" not in content_type:
        return {
            "crawl_status": "NOT_HTML",
            "http_status": status,
            "body_length": 0,
            "body": None,
            "crawl_error": f"Content-Type: {content_type}",
        }

    try:
        body = extract(
            response.text,
            url=response.url,
            include_comments=False,
            include_tables=False,
        )

    except Exception as e:
        return {
            "crawl_status": "EXTRACT_ERROR",
            "http_status": status,
            "body_length": 0,
            "body": None,
            "crawl_error": f"{type(e).__name__}: {e}",
        }

    if not body:
        return {
            "crawl_status": "EXTRACT_FAILED",
            "http_status": status,
            "body_length": 0,
            "body": None,
            "crawl_error": "Trafilatura returned None",
        }

    body = body.strip()
    body_length = len(body)

    # 너무 짧으면 실제 기사 본문이라고 보기 어려움
    if body_length < min_body_length:
        return {
            "crawl_status": "CONTENT_TOO_SHORT",
            "http_status": status,
            "body_length": body_length,
            "body": body,
            "crawl_error": (
                f"Body length < {min_body_length}"
            ),
        }

    return {
        "crawl_status": "SUCCESS",
        "http_status": status,
        "body_length": body_length,
        "body": body,
        "crawl_error": None,
    }

In [20]:
def run_crawlability_test(
    test_articles,
    timeout=15,
    min_body_length=300,
):
    results = []

    session = requests.Session()

    session.headers.update({
        "User-Agent": (
            "Mozilla/5.0 "
            "(Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 "
            "(KHTML, like Gecko) "
            "Chrome/140.0 Safari/537.36"
        ),
        "Accept-Language": "en-US,en;q=0.9",
    })

    total = len(test_articles)

    for i, article in enumerate(
        test_articles,
        start=1,
    ):
        result = crawl_article_for_test(
            article=article,
            session=session,
            timeout=timeout,
            min_body_length=min_body_length,
        )

        record = {
            "requested_source": article.get(
                "requested_source"
            ),
            "query": article.get("query"),
            "title": article.get("title"),
            "original_url": article.get(
                "original_url"
            ),
            **result,
        }

        results.append(record)

        print(
            f"[{i}/{total}] "
            f"{article.get('requested_source')} | "
            f"{article.get('query')} | "
            f"{result['crawl_status']} "
            f"({result.get('http_status')})"
        )

    session.close()

    return results

In [21]:
test_results = run_crawlability_test(
    test_articles,
    timeout=15,
    min_body_length=300,
)

[1/893] finance.yahoo.com | stock market | SUCCESS (200)
[2/893] finance.yahoo.com | stock market | SUCCESS (200)
[3/893] finance.yahoo.com | stock market | SUCCESS (200)
[4/893] finance.yahoo.com | stock market | SUCCESS (200)
[5/893] finance.yahoo.com | stock market | SUCCESS (200)
[6/893] finance.yahoo.com | stock market | SUCCESS (200)
[7/893] finance.yahoo.com | stock market | SUCCESS (200)
[8/893] finance.yahoo.com | stock market | SUCCESS (200)
[9/893] finance.yahoo.com | stock market | SUCCESS (200)
[10/893] finance.yahoo.com | stock market | SUCCESS (200)
[11/893] cnbc.com | stock market | SUCCESS (200)
[12/893] cnbc.com | stock market | SUCCESS (200)
[13/893] cnbc.com | stock market | SUCCESS (200)
[14/893] cnbc.com | stock market | SUCCESS (200)
[15/893] cnbc.com | stock market | SUCCESS (200)
[16/893] cnbc.com | stock market | SUCCESS (200)
[17/893] cnbc.com | stock market | CONTENT_TOO_SHORT (200)
[18/893] cnbc.com | stock market | SUCCESS (200)
[19/893] cnbc.com | stock m

In [22]:
detail_df = pd.DataFrame(test_results)

detail_df

,requested_source,query,title,original_url,crawl_status,http_status,body_length,body,crawl_error
0,finance.yahoo.com,stock market,Annaly Capital Management (NLY) Surpasses Q3 E...,https://finance.yahoo.com/markets/stocks/artic...,SUCCESS,200.0,2734,Annaly Capital Management (NLY) came out with ...,None
1,finance.yahoo.com,stock market,Qualcomm (QCOM) Beats Stock Market Upswing: Wh...,https://finance.yahoo.com/markets/stocks/artic...,SUCCESS,200.0,2706,Qualcomm (QCOM) ended the recent trading sessi...,None
2,finance.yahoo.com,stock market,The Market Is Triggering a Warning Not Seen in...,https://finance.yahoo.com/markets/stocks/artic...,SUCCESS,200.0,2823,The market has posted one of its best-ever str...,None
3,finance.yahoo.com,stock market,Could Quantinuum's Breakthrough Make Its Sell-...,https://finance.yahoo.com/markets/stocks/artic...,SUCCESS,200.0,7426,Key Points \n- Interested in Quantinuum Inc.?...,None
4,finance.yahoo.com,stock market,Trump announces $8 billion for farmers to offs...,https://finance.yahoo.com/markets/article/trum...,SUCCESS,200.0,1986,President Trump announced an $8 billion aid pa...,None
...,...,...,...,...,...,...,...,...,...
888,apnews.com,Federal Reserve,UK inflation rises to highest in almost a year...,https://apnews.com/article/britain-inflation-i...,HTTP_403,403.0,0,None,HTTP 403
889,apnews.com,earnings,Meta Q3 revenue tops estimates and Zuckerberg ...,https://apnews.com/article/meta-earnings-faceb...,HTTP_403,403.0,0,None,HTTP 403
890,apnews.com,earnings,Meta ordered to pay New Mexico $25 billion for...,https://apnews.com/article/facebook-penalty-ca...,HTTP_403,403.0,0,None,HTTP 403
891,foxbusiness.com,NVIDIA,This is the way to deploy the latest Nvidia te...,https://www.foxbusiness.com/video/6407194365112,SUCCESS,200.0,399,Live Now\nAll times eastern\nNOW - 3:30 AM\n3:...,None


In [23]:
def make_query_summary(detail_df):

    base = (
        detail_df
        .groupby(
            ["requested_source", "query"]
        )
        .agg(
            total=(
                "crawl_status",
                "size",
            ),
            success=(
                "crawl_status",
                lambda x: (
                    x == "SUCCESS"
                ).sum(),
            ),
            avg_body_length=(
                "body_length",
                lambda x: (
                    x[x > 0].mean()
                    if (x > 0).any()
                    else 0
                ),
            ),
        )
        .reset_index()
    )

    base["success_rate"] = (
        base["success"]
        / base["total"]
    )

    # 상태별 개수
    status_counts = (
        pd.crosstab(
            [
                detail_df["requested_source"],
                detail_df["query"],
            ],
            detail_df["crawl_status"],
        )
        .reset_index()
    )

    summary = base.merge(
        status_counts,
        on=[
            "requested_source",
            "query",
        ],
        how="left",
    )

    return summary.sort_values(
        [
            "requested_source",
            "query",
        ]
    ).reset_index(drop=True)

In [25]:
def make_source_summary(detail_df):

    base = (
        detail_df
        .groupby("requested_source")
        .agg(
            total=(
                "crawl_status",
                "size",
            ),
            success=(
                "crawl_status",
                lambda x: (
                    x == "SUCCESS"
                ).sum(),
            ),
            avg_body_length=(
                "body_length",
                lambda x: (
                    x[x > 0].mean()
                    if (x > 0).any()
                    else 0
                ),
            ),
        )
        .reset_index()
    )

    base["success_rate"] = (
        base["success"]
        / base["total"]
    )

    status_counts = (
        pd.crosstab(
            detail_df["requested_source"],
            detail_df["crawl_status"],
        )
        .reset_index()
    )

    summary = base.merge(
        status_counts,
        on="requested_source",
        how="left",
    )

    return summary.sort_values(
        "success_rate",
        ascending=False,
    ).reset_index(drop=True)

In [24]:
query_summary = make_query_summary(
    detail_df
)

query_summary

,requested_source,query,total,success,avg_body_length,success_rate,CONTENT_TOO_SHORT,HTTP_401,HTTP_403,HTTP_412,NOT_HTML,REQUEST_ERROR,SUCCESS
0,apnews.com,Federal Reserve,2,0,0.000000,0.0,0,0,2,0,0,0,0
1,apnews.com,NVIDIA,1,0,0.000000,0.0,0,0,1,0,0,0,0
2,apnews.com,Wall Street,10,0,0.000000,0.0,0,0,10,0,0,0,0
3,apnews.com,earnings,2,0,0.000000,0.0,0,0,2,0,0,0,0
4,apnews.com,stock market,7,0,0.000000,0.0,0,0,7,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,zacks.com,Federal Reserve,3,3,7137.666667,1.0,0,0,0,0,0,0,3
95,zacks.com,NVIDIA,10,10,4387.000000,1.0,0,0,0,0,0,0,10
96,zacks.com,Wall Street,10,10,4741.900000,1.0,0,0,0,0,0,0,10
97,zacks.com,earnings,10,10,5391.400000,1.0,0,0,0,0,0,0,10


In [26]:
source_summary = make_source_summary(
    detail_df
)

source_summary

,requested_source,total,success,avg_body_length,success_rate,CONTENT_TOO_SHORT,HTTP_401,HTTP_403,HTTP_412,NOT_HTML,REQUEST_ERROR,SUCCESS
0,zacks.com,43,43,4825.023256,1.000000,0,0,0,0,0,0,43
1,cnn.com,33,33,19772.575758,1.000000,0,0,0,0,0,0,33
2,foxbusiness.com,29,29,2002.034483,1.000000,0,0,0,0,0,0,29
3,fool.com,40,40,19389.575000,1.000000,0,0,0,0,0,0,40
4,businessinsider.com,50,50,6422.240000,1.000000,0,0,0,0,0,0,50
5,finance.yahoo.com,50,50,3463.720000,1.000000,0,0,0,0,0,0,50
6,fortune.com,48,47,4932.021277,0.979167,0,0,0,0,1,0,47
7,cnbc.com,50,48,3708.300000,0.960000,2,0,0,0,0,0,48
8,ft.com,50,31,6342.806452,0.620000,0,0,19,0,0,0,31
9,forbes.com,48,10,4632.909091,0.208333,1,0,37,0,0,0,10


In [27]:
USABLE_THRESHOLD = 0.8

usable_sources = (
    source_summary.loc[
        source_summary["success_rate"]
        >= USABLE_THRESHOLD,
        "requested_source",
    ]
    .tolist()
)

usable_sources

['zacks.com',
 'cnn.com',
 'foxbusiness.com',
 'fool.com',
 'businessinsider.com',
 'finance.yahoo.com',
 'fortune.com',
 'cnbc.com']

In [49]:
SOURCE = {
    # 1순위: 미국 증시/금융 전문
    "finance.yahoo.com": "Yahoo Finance",
    "cnbc.com": "CNBC",
 
    # 2순위: 경제/비즈니스 종합
    "fortune.com": "Fortune",
    "businessinsider.com": "Business Insider",
    "cnn.com": "CNN Business",
    "foxbusiness.com": "Fox Business",

    # 3순위: 투자 전문
    #"benzinga.com": "Benzinga",
    "fool.com": "The Motley Fool",
    "zacks.com": "Zacks",
}

In [1]:
# 딕셔너리의 키값들만 추출하여 필터링
filtered_df = test_results[test_results['requested_source'].isin(SOURCE.keys())]

NameError: name 'test_results' is not defined

In [ ]:
for index, row in filtered_df.iterrows():
    print(f"--- 행 번호: {index} ---")
    print(f"언론사: {row['requested_source']}")
    print(f"제목: {row['title']}")
    print(f"URL: {row['original_url']}")
    print(f"본문 일부: {str(row['body'])[:500]}...") # 본문이 길므로 앞 100자만 출력
    print("\n" + "="*50 + "\n")